# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [1]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [3]:
# Task 1
ratings_to_show = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']

df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

rating_decade_counts = (
    df[df['rating'].isin(ratings_to_show)]
    .pivot_table(
        index='rating',
        columns='decade',
        values='type',
        aggfunc='count',
        fill_value=0,
    )
    .reindex(ratings_to_show)
    .reindex(sorted(df['decade'].dropna().unique()), axis=1, fill_value=0)
)

max_rating = rating_decade_counts.stack().idxmax()[0]
max_decade = rating_decade_counts.stack().idxmax()[1]
max_titles = rating_decade_counts.loc[max_rating, max_decade]

fig = go.Figure(
    data=go.Heatmap(
        z=rating_decade_counts.values,
        x=rating_decade_counts.columns.tolist(),
        y=rating_decade_counts.index.tolist(),
        colorscale='Blues',
        colorbar={'title': 'Titles'},
        hovertemplate='Rating: %{y}<br>Decade: %{x}<br>Titles: %{z}<extra></extra>',
    )
)

for rating in rating_decade_counts.index:
    for decade in rating_decade_counts.columns:
        value = rating_decade_counts.loc[rating, decade]
        fig.add_annotation(
            x=decade,
            y=rating,
            text=str(value),
            showarrow=False,
            font={'color': 'white' if value > max_titles * 0.55 else '#17324d'},
        )

fig.add_annotation(
    x=max_decade,
    y=max_rating,
    text='Peak cell',
    showarrow=True,
    arrowhead=2,
    ax=45,
    ay=-45,
    bgcolor='white',
    bordercolor='#1f77b4',
)

fig.update_layout(
    title=f'Netflix ratings cluster around {max_rating} in the {max_decade}, with {max_titles} titles',
    template='plotly_white',
    title_x=0.5,
    xaxis_title='Release decade',
    yaxis_title='Content rating',
    height=500,
)

fig.show()

print('Task 1 Output')
print(rating_decade_counts.to_string())
print(f'Peak cell: {max_rating} in the {max_decade} = {max_titles} titles')


Task 1 Output
decade  2000s  2010s  2020s
rating                     
TV-14     296    295    142
TV-MA     348    359    133
PG-13     259    227    103
R         132    126     54
PG         75     81     40
Peak cell: TV-MA in the 2010s = 359 titles


## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [4]:
# Task 2
movie_additions = (
    df[(df['type'] == 'Movie') & (df['added_year'].between(2015, 2022))]
    .groupby('added_year')
    .size()
    .reindex(range(2015, 2023), fill_value=0)
)

largest_year = int(movie_additions.idxmax())
largest_addition = int(movie_additions.max())
cumulative_movies = movie_additions.cumsum()
cumulative_at_largest = int(cumulative_movies.loc[largest_year])
total_movies = int(movie_additions.sum())

fig = go.Figure(
    go.Waterfall(
        name='Movie additions',
        orientation='v',
        measure=['relative'] * len(movie_additions) + ['total'],
        x=[str(year) for year in movie_additions.index] + ['2015-2022 total'],
        y=movie_additions.tolist() + [None],
        text=movie_additions.astype(str).tolist() + [str(total_movies)],
        textposition='outside',
        connector={'line': {'color': 'rgba(70, 70, 70, 0.35)'}},
        increasing={'marker': {'color': '#2ca02c'}},
        decreasing={'marker': {'color': '#d62728'}},
        totals={'marker': {'color': '#1f77b4'}},
    )
)

fig.add_annotation(
    x=str(largest_year),
    y=cumulative_at_largest,
    text=f'Largest jump: {largest_addition} movies in {largest_year}',
    showarrow=True,
    arrowhead=2,
    ax=55,
    ay=-55,
    bgcolor='white',
    bordercolor='#2ca02c',
)

fig.update_layout(
    title=f'Netflix movie catalogue kept climbing: {total_movies} movies added from 2015 to 2022',
    title_x=0.5,
    template='plotly_white',
    xaxis_title='Year added',
    yaxis_title='Number of movies',
    height=550,
    showlegend=False,
)

fig.show()

print('Task 2 Output')
print(movie_additions.to_string())
print(f'Cumulative total by 2022: {total_movies}')
print(f'Largest movie addition year: {largest_year} = {largest_addition} movies')
print(f'Total movies added from 2015 to 2022: {total_movies}')


Task 2 Output
added_year
2015    71
2016    93
2017    77
2018    79
2019    93
2020    81
2021    83
2022    82
Cumulative total by 2022: 659
Largest movie addition year: 2016 = 93 movies
Total movies added from 2015 to 2022: 659
